# 03 — The Data Model

**Purpose:** Deep-dive tour of every entity type in a `Trace`.  One section per record
class: `Op`, `Layer`, `Module`, `ModuleCall`, `Param`, `Buffer`, `GradFn`, `GradFnCall`.
Each section shows `__repr__`, `to_pandas()`, and the key inspectable fields.

**Surfaces covered:**
- [ ] `Op.__repr__` + `Op.to_pandas()`
- [ ] `Layer.__repr__` + `Layer.to_pandas()`
- [ ] `Module.__repr__` + `Module.to_pandas()`
- [ ] `ModuleCall.__repr__` + `ModuleCall.to_pandas()`
- [ ] `Param.__repr__` + `Param.to_pandas()`
- [ ] `Buffer.__repr__` + `Buffer.to_pandas()`
- [ ] `GradFn.__repr__` + `GradFn.to_pandas()`
- [ ] `GradFnCall.__repr__` + `GradFnCall.to_pandas()`
- [ ] `trace.module_calls` — `TraceModuleCallAccessor` repr
- [ ] `trace.grad_fns` — `GradFnAccessor` repr
- [ ] Key navigational properties: `op.parents`, `op.children`, `op.params`, `op.module`
- [ ] `Layer.ops` — the per-pass Op list within a Layer

## 1. Environment setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torch.nn as nn
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. Traces used in this notebook

- `trace_mlp` — `tiny_mlp` (8->relu->4): simple trace for Op/Layer/Module/ModuleCall/Param
- `trace_bn` — `batch_norm` model: exposes `Buffer` records
- `trace_bwd` — a `SumModel` with `.backward()` called: exposes `GradFn` / `GradFnCall`

In [ ]:
# tiny_mlp trace
model_mlp, x_mlp = ZOO["tiny_mlp"]()
trace_mlp = tl.trace(model_mlp, x_mlp)
print("trace_mlp :", repr(trace_mlp))

# batch_norm trace (for buffers)
model_bn, x_bn = ZOO["batch_norm"]()
trace_bn = tl.trace(model_bn, x_bn)
print("trace_bn  :", repr(trace_bn))

# SumModel trace with backward (for GradFn / GradFnCall)


class SumModel(nn.Module):
    """Tiny model that collapses to a scalar for backward convenience."""

    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(self.fc(x)).sum()


m_sum = SumModel()
x_sum = torch.randn(2, 4, requires_grad=True)
trace_bwd = tl.trace(m_sum, x_sum)
# Run backward via the captured output tensor
trace_bwd[-1].out.backward()
print("trace_bwd :", repr(trace_bwd))
print("grad_fns  :", trace_bwd.grad_fns)

---

## 3. `Op` — single execution record

An `Op` represents **one execution** of a torch function during the forward pass.
Integer indexing (`trace[int]`) and op-label indexing (`trace["relu_1_2:1"]`) return `Op`.
Key fields: `layer_label`, `raw_label`, `func_name`, `shape`, `dtype`, `out`,
`activation_memory`, `func_duration`, `flops_forward`, `parents`, `children`, `params`, `module`.

In [ ]:
op = trace_mlp["linear_1_1"]
print("=== Op.__repr__ ===")
print(repr(op))

In [ ]:
# Key navigational fields
print("op.func_name     :", op.func_name)
print("op.layer_label   :", op.layer_label)
print("op.raw_label     :", op.raw_label)
print("op.shape         :", op.shape)
print("op.dtype         :", op.dtype)
print("op.parents       :", op.parents)
print("op.children      :", op.children)
print("op.params        :", op.params)
print("op.module        :", op.module)
print("op.in_submodule  :", op.in_submodule)

In [ ]:
print("=== Op.to_pandas() — shape + a few key columns ===")
df_op = op.to_pandas()
print(f"Shape: {df_op.shape}  (1 row x {df_op.shape[1]} columns)")
key_op_cols = [
    "layer_label",
    "func_name",
    "shape",
    "dtype",
    "activation_memory",
    "flops_forward",
    "macs_forward",
    "is_input",
    "is_output",
    "in_submodule",
]
print(df_op[[c for c in key_op_cols if c in df_op.columns]].to_string())

---

## 4. `Layer` — all-passes group record

A `Layer` groups all `Op` instances that share the same logical label across loop passes.
For non-recurrent models each `Layer` has exactly one `Op`.  
Layer-label indexing (`trace["relu_1_2"]`) returns a `Layer`.
Key fields: `layer_label`, `num_passes`, `ops`, `shape`, `total_activation_memory`,
`total_flops_forward`, `parents`, `children`, `params`, `module`.

In [ ]:
# NOTE: trace[label] returns Op (single execution); trace.layers[label] returns Layer (all passes)
# Use trace.layers accessor to get the Layer record
layer = trace_mlp.layers["linear_1_1"]
print("=== Layer.__repr__ ===")
print(repr(layer))

In [ ]:
print("layer.layer_label              :", layer.layer_label)
print("layer.num_passes               :", layer.num_passes)
print("layer.shape                    :", layer.shape)
print("layer.total_activation_memory  :", layer.total_activation_memory, "B")
print("layer.total_flops_forward      :", layer.total_flops_forward, "FLOPs")
print("layer.parents                  :", layer.parents)
print("layer.children                 :", layer.children)

# Layer.ops — use .op_labels then index into trace
print()
print("layer.op_labels :", layer.op_labels)
print("layer.ops       :", layer.ops)

In [ ]:
print("=== Layer.to_pandas() — shape + key columns ===")
df_layer = layer.to_pandas()
print(f"Shape: {df_layer.shape}")
# Note: in the DataFrame the per-layer aggregated columns are named without "total_" prefix
key_layer_cols = [
    "layer_label",
    "layer_type",
    "shape",
    "dtype",
    "num_passes",
    "activation_memory",
    "flops_forward",
    "is_input",
    "is_output",
]
print(df_layer[[c for c in key_layer_cols if c in df_layer.columns]].to_string())

# Multi-pass Layer (from demo_model with loop submodule)
print()
print("--- Multi-pass Layer example (demo_model loop) ---")
model_d, x_d = ZOO["demo_model"]()
trace_d = tl.trace(model_d, x_d)
# Use trace.layers accessor to check num_passes (Layer property)
multi_labels = [lbl for lbl in trace_d.layer_labels if trace_d.layers[lbl].num_passes > 1]
print("Labels with >1 pass:", multi_labels)
if multi_labels:
    lmp = trace_d.layers[multi_labels[0]]
    print(repr(lmp))

---

## 5. `Module` — `nn.Module` node record

A `Module` record describes one `nn.Module` in the network's module hierarchy.
Access via `trace.modules[address]`.  Key fields: `address`, `class_name`,
`call_depth`, `address_depth`, `num_params`, `num_layers`, `num_calls`,
`call_labels`, `params`, `layers`, `total_flops`, `total_forward_duration`.

In [ ]:
print("=== trace.modules ===")
print(repr(trace_mlp.modules))

mod = trace_mlp.modules["in_proj"]
print()
print("=== Module.__repr__ for 'in_proj' ===")
print(repr(mod))

In [ ]:
print("mod.address          :", mod.address)
print("mod.class_name       :", mod.class_name)
print("mod.call_depth       :", mod.call_depth)
print("mod.address_depth    :", mod.address_depth)
print("mod.num_params       :", mod.num_params)
print("mod.num_layers       :", mod.num_layers)
print("mod.num_calls        :", mod.num_calls)
print("mod.call_labels      :", mod.call_labels)
print("mod.layer_labels     :", mod.layer_labels)
print("mod.total_flops      :", mod.total_flops)
print("mod.total_forward_duration :", mod.total_forward_duration)

In [ ]:
print("=== Module.to_pandas() ===")
df_mod = mod.to_pandas()
print(f"Shape: {df_mod.shape}")
print(f"Columns: {df_mod.columns.tolist()}")
print()
# Note: Module.to_pandas returns a row per layer INSIDE this module, not one row per module
print(df_mod.to_string())

---

## 6. `ModuleCall` — one invocation of a module

A `ModuleCall` represents one call instance of an `nn.Module`.  If the same module is
called N times in a loop, there are N `ModuleCall` records.  Access via
`module.calls[call_label]` (e.g. `"in_proj:1"`) or `trace.module_calls[call_label]`.  
Key fields: `address`, `call_index`, `call_label`, `num_ops`, `input_layers`,
`output_layers`, `out_shape`, `forward_duration`.

In [ ]:
print("=== trace.module_calls ===")
print(repr(trace_mlp.module_calls))

# Access via module.calls[call_label]
mod_calls = trace_mlp.modules["in_proj"].calls
call_lbl = trace_mlp.modules["in_proj"].call_labels[0]  # 'in_proj:1'
mc = mod_calls[call_lbl]

print()
print(f"=== ModuleCall.__repr__ for '{call_lbl}' ===")
print(repr(mc))

In [ ]:
print("mc.address       :", mc.address)
print("mc.call_index    :", mc.call_index)
print("mc.call_label    :", mc.call_label)
print("mc.num_ops       :", mc.num_ops)
print("mc.input_layers  :", mc.input_layers)
print("mc.output_layers :", mc.output_layers)
print("mc.out_shape     :", mc.out_shape)
print("mc.forward_duration :", mc.forward_duration)

In [ ]:
print("=== ModuleCall.to_pandas() ===")
df_mc = mc.to_pandas()
print(f"Shape: {df_mc.shape}")
key_mc_cols = [
    "address",
    "name",
    "call_index",
    "call_label",
    "num_ops",
    "input_layers",
    "output_layers",
]
print(df_mc[[c for c in key_mc_cols if c in df_mc.columns]].to_string())

---

## 7. `Param` — parameter tensor record

A `Param` record describes one parameter tensor (`nn.Parameter`) that participates in
the trace.  Access via `trace.params[address]` (e.g. `"in_proj.weight"`).  
Key fields: `address`, `name`, `shape`, `dtype`, `is_trainable`, `module_address`,
`param_memory`, `num_params`, `value`.

In [ ]:
print("=== trace.params ===")
print(repr(trace_mlp.params))

param = trace_mlp.params["in_proj.weight"]
print()
print("=== Param.__repr__ for 'in_proj.weight' ===")
print(repr(param))

In [ ]:
print("param.address        :", param.address)
print("param.name           :", param.name)
print("param.shape          :", param.shape)
print("param.dtype          :", param.dtype)
print("param.is_trainable   :", param.is_trainable)
print("param.module_address :", param.module_address)
print("param.param_memory   :", param.param_memory, "B")
print("param.num_params     :", param.num_params)
print("param.has_grad       :", param.has_grad)
print("param.value type     :", type(param.value).__name__)

In [ ]:
print("=== Param.to_pandas() ===")
df_param = param.to_pandas()
print(f"Shape: {df_param.shape}")
key_p_cols = [
    "address",
    "name",
    "shape",
    "dtype",
    "num_params",
    "param_memory",
    "is_trainable",
    "has_grad",
]
print(df_param[[c for c in key_p_cols if c in df_param.columns]].to_string())

---

## 8. `Buffer` — buffer tensor record

A `Buffer` record describes one `register_buffer` tensor accessed during the forward
pass.  Use `batch_norm` which has `running_mean`, `running_var`, and
`num_batches_tracked`.  Access via `trace.buffers[address]`.  
Key fields: `address`, `name`, `shape`, `dtype`, `buffer_pass`, `buffer_overwrite_index`,
`activation_memory`, `has_saved_activation`.

In [ ]:
print("=== trace.buffers (batch_norm) ===")
print(repr(trace_bn.buffers))

buf_key = list(trace_bn.buffers.keys())[0]
buf = trace_bn.buffers[buf_key]
print()
print(f"=== Buffer.__repr__ for '{buf_key}' ===")
print(repr(buf))

In [ ]:
print("buf.address              :", buf.address)
print("buf.name                 :", buf.name)
print("buf.shape                :", buf.shape)
print("buf.dtype                :", buf.dtype)
print("buf.buffer_pass          :", buf.buffer_pass)
print("buf.buffer_overwrite_index:", buf.buffer_overwrite_index)
print("buf.activation_memory    :", buf.activation_memory, "B")
print("buf.has_saved_activation :", buf.has_saved_activation)

print()
print("All buffers in batch_norm trace:")
for k, b in trace_bn.buffers.items():
    print(f"  {k}: shape={b.shape} dtype={b.dtype} overwrites={b.buffer_overwrite_index}")

In [ ]:
print("=== Buffer.to_pandas() ===")
df_buf = buf.to_pandas()
print(f"Shape: {df_buf.shape}")
key_b_cols = [
    "address",
    "name",
    "shape",
    "dtype",
    "activation_memory",
    "has_saved_activation",
    "has_grad",
]
print(df_buf[[c for c in key_b_cols if c in df_buf.columns]].to_string())

---

## 9. `GradFn` — autograd node record

A `GradFn` record describes one autograd function node (e.g. `ReluBackward0`) from the
backward graph.  `GradFn` records are populated **after** calling `.backward()` on the
output.  Access via `trace.grad_fns[label]`.  
Key fields: `label`, `cls`, `class_name`, `has_op`, `op_label`, `order`,
`parents`, `children`, `num_calls`.

In [ ]:
print("=== trace_bwd.grad_fns ===")
print(repr(trace_bwd.grad_fns))

In [ ]:
# Show a grad_fn that has a paired forward Op (has_op=True)
gfn_with_op = next(gf for gf in trace_bwd.grad_fns.values() if gf.has_op)
print(f"=== GradFn.__repr__ for '{gfn_with_op.label}' ===")
print(repr(gfn_with_op))

In [ ]:
print("gfn.label        :", gfn_with_op.label)
print("gfn.class_name   :", gfn_with_op.class_name)
print("gfn.has_op       :", gfn_with_op.has_op)
print("gfn.op_label     :", gfn_with_op.op_label)
print("gfn.order        :", gfn_with_op.order)
print("gfn.num_calls    :", gfn_with_op.num_calls)
print("gfn.parents      :", gfn_with_op.parents)
print("gfn.children     :", gfn_with_op.children)

In [ ]:
print("=== GradFn.to_pandas() ===")
df_gf = gfn_with_op.to_pandas()
print(f"Shape: {df_gf.shape}")
key_gf_cols = [
    "grad_fn_object_id",
    "class_name",
    "class_qualname",
    "class_source_file",
    "class_source_line",
]
print(df_gf[[c for c in key_gf_cols if c in df_gf.columns]].to_string())

---

## 10. `GradFnCall` — one backward invocation record

A `GradFnCall` records one backward call through a `GradFn` node.  
Access via `grad_fn.calls`.  Key fields: `call_index`, `ordinal`,
`backward_pass_index`, `call_label`, `backward_duration`, `grad_inputs`, `grad_outputs`.

In [ ]:
# GradFnCallAccessor: use .values() to get GradFnCall objects (iterating yields ints like ModuleCallAccessor)
gfn_calls = gfn_with_op.calls
print("gfn.calls type :", type(gfn_calls).__name__)
print("call keys      :", list(gfn_calls.keys()))

gfn_call = list(gfn_calls.values())[0]
print()
print("=== GradFnCall.__repr__ ===")
print(repr(gfn_call))

In [ ]:
print("gfn_call.call_index         :", gfn_call.call_index)
print("gfn_call.ordinal            :", gfn_call.ordinal)
print("gfn_call.backward_pass_index:", gfn_call.backward_pass_index)
print("gfn_call.call_label         :", gfn_call.call_label)
print("gfn_call.backward_duration  :", gfn_call.backward_duration)
print("gfn_call.grad_inputs        :", gfn_call.grad_inputs)  # None unless save_grads
print("gfn_call.grad_outputs       :", gfn_call.grad_outputs)

In [ ]:
print("=== GradFnCall.to_pandas() ===")
df_gfc = gfn_call.to_pandas()
print(f"Shape: {df_gfc.shape}")
print(f"Columns: {df_gfc.columns.tolist()}")
print()
print(df_gfc.to_string())

---

## 11. Cross-record navigation summary

Quick illustration of how the entity types link to each other.

In [ ]:
op = trace_mlp["linear_1_1"]

print("Starting from Op 'linear_1_1':")
print("  op.layer_label   ->", op.layer_label)  # str
print("  op.parents       ->", op.parents)  # list of layer labels
print("  op.children      ->", op.children)  # list of layer labels
print("  op.params        ->", op.params)  # dict of Param records
print("  op.module        ->", op.module)  # module call label (str), e.g. 'in_proj:1'
print("  op.in_submodule  ->", op.in_submodule)  # bool
print()

# Navigate to parent Op
parent_lbl = op.parents[0]
parent_op = trace_mlp[parent_lbl]
print(f"Parent op '{parent_lbl}':")
print("  type  :", type(parent_op).__name__)
print("  shape :", parent_op.shape)
print()

# Navigate to Module object via trace.modules + op.atomic_module_address
module_addr = op.atomic_module_address
print(f"op.atomic_module_address: '{module_addr}'")
if module_addr and module_addr in trace_mlp.modules:
    mod_obj = trace_mlp.modules[module_addr]
    print(f"trace_mlp.modules['{module_addr}']:")
    print("  type       :", type(mod_obj).__name__)
    print("  address    :", mod_obj.address)
    print("  num_params :", mod_obj.num_params)
else:
    print("  (no module or module not in trace.modules)")

---

## ⚠️ GAPs / ergonomic smells

- **`trace[label]` returns `Op`, NOT `Layer`**, even when using a layer label.\n",
  `trace[\"linear_1_1\"]` returns an `Op` object (type `Op`).  To get the `Layer` record\n",
  (all-passes group, `num_passes`, `total_activation_memory`, etc.) you must use\n",
  `trace.layers[\"linear_1_1\"]`.  The section 4 markdown says \"Layer-label indexing returns\n",
  a `Layer`\" but that is NOT true for single-pass layers in the current implementation.\n",
  This inconsistency is a significant ergonomic trap for new users.\n",

- **`Layer.total_activation_memory` / `Layer.total_flops_forward` are live properties**\n",
  but the corresponding `to_pandas()` columns are named without the `total_` prefix\n",
  (`activation_memory`, `flops_forward`).  Property name vs DataFrame column name mismatch.\n",

- **`Module.to_pandas()` returns one row per LAYER inside the module**, not one row per\n",
  module.  This is unexpected: calling `mod.to_pandas()` on `in_proj` gives 1 row (the\n",
  one layer inside it), but calling it on the root `self` module gives 5 rows (all layers).\n",
  A new user would expect `to_pandas()` to return a single-row summary of the module's own\n",
  identity fields (address, class, depths, param count), not its contained layers.\n",

- **`ModuleCallAccessor.__iter__` yields call-index integers, not `ModuleCall` objects.**\n",
  `list(module.calls)` returns `[1]` (the integer call index), NOT `[ModuleCall(...)]`.\n",
  To get the `ModuleCall` object you must use `module.calls[call_label]` or\n",
  `module.calls.values()`.  Same pattern applies to `GradFnCallAccessor`.\n",
  `for call in module.calls: ...` silently yields ints — a discovery trap.\n",

- **`GradFnCall.grad_inputs` and `.grad_outputs` are `None` by default.**  They are\n",
  only populated when `save_grads=True` is passed to `tl.trace`.  There is no hint in\n",
  the `GradFnCall` repr that this flag exists.  Discoverability is low.\n",

- **`Param` has no `.size` or `.tensor` convenience property.**  `param.value` gives\n",
  the underlying tensor, but `param.size` raises `AttributeError`.\n",

- **`GradFn` records are empty until `.backward()` is called on the output tensor.**\n",
  `trace.grad_fns` returns `GradFnAccessor({})` immediately after `tl.trace(...)`.  This\n",
  is correct (no backward has run yet), but the empty dict repr gives no hint that\n",
  these records will appear after backward.  A message like `GradFnAccessor(0 items —\n",
  run .backward() to populate)` would help discoverability.